In [2]:
#install
!pip install -q gradio grad-cam timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 81.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
#imports and setup
import gradio as gr
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import timm
from pathlib import Path
from PIL import Image
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from google.colab import drive
drive.mount('/content/drive')

device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = Path('/content/drive/MyDrive/glenda_splits')

print(f"Device: {device}")

# ── Colour palette ────────────────────────────────────────────────────────────
COLOUR_PALETTE = [
    (0,   0,   0),
    (190, 62, 204),   # Peritoneum  — purple
    (93,  89, 254),   # Ovary       — blue
    (145, 210, 138),  # TIE         — green
    (238, 236, 50),   # Uterus      — yellow
]
CLASS_COLOURS = {
    'Endo-Peritoneum' : (190, 62,  204),
    'Endo-Ovary'      : (93,  89,  254),
    'Endo-TIE'        : (145, 210, 138),
    'Endo-Uterus'     : (238, 236, 50),
}

eval_tf = T.Compose([
    T.Resize((320, 320), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

# ── Load models ───────────────────────────────────────────────────────────────
def load_model(ckpt_path):
    model = timm.create_model('edgenext_small', pretrained=False, num_classes=2)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.to(device)
    model.eval()
    return model

# Use best fold (fold 1) for both models
general_model = load_model(OUTPUT_DIR / 'edgenext_small' / 'fold1_best.pt')
spe_model     = load_model(OUTPUT_DIR / 'edgenext_small_peritoneal' / 'fold1_peritoneal_best.pt')
print("Both models loaded")

# ── Helpers ───────────────────────────────────────────────────────────────────
def parse_mask_to_binary(annot_path: str) -> np.ndarray:
    mask   = np.array(Image.open(annot_path).convert('RGB'))
    binary = np.zeros(mask.shape[:2], dtype=np.uint8)
    for c in COLOUR_PALETTE[1:]:
        binary[np.all(mask == c, axis=-1)] = 1
    return binary

def build_gt_overlay(img_np: np.ndarray, annot_path: str) -> np.ndarray:
    """
    Build a four-colour GT mask overlay.
    Each lesion class shown in its original annotation colour.
    """
    mask_rgb  = np.array(Image.open(annot_path).convert('RGB').resize(
        (320, 320), Image.NEAREST
    ))
    overlay   = img_np.copy().astype(np.float32)
    for name, colour in CLASS_COLOURS.items():
        region = np.all(mask_rgb == colour, axis=-1)
        if region.sum() > 0:
            overlay[region] = (
                overlay[region] * 0.4 +
                np.array(colour, dtype=np.float32) * 0.6
            )
    return np.clip(overlay, 0, 255).astype(np.uint8)

def run_inference(model, img_pil: Image.Image):
    """Returns prob_pathological, img_np_320, tensor."""
    img_pil_320 = img_pil.resize((320, 320), Image.BICUBIC)
    img_np      = np.array(img_pil_320)
    tensor      = eval_tf(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tensor)
        probs  = F.softmax(logits, dim=1)[0]
    prob_normal = probs[0].item()
    prob_patho  = probs[1].item()
    return prob_normal, prob_patho, img_np, tensor

def run_gradcampp(model, tensor, img_np):
    """Returns normalised GradCAM++ heatmap and overlay."""
    cam     = GradCAMPlusPlus(model=model, target_layers=[model.stages[-1]])
    targets = [ClassifierOutputTarget(1)]
    gc      = cam(input_tensor=tensor, targets=targets)[0]
    gc_norm = (gc - gc.min()) / (gc.max() - gc.min() + 1e-8)
    overlay = show_cam_on_image(
        img_np.astype(np.float32) / 255.0, gc_norm, use_rgb=True
    )
    return gc_norm, overlay

def make_confidence_bar(prob: float, label: str) -> str:
    """Returns an HTML confidence bar."""
    pct   = int(prob * 100)
    color = '#e74c3c' if label == 'Pathological' else '#2ecc71'
    return f"""
    <div style='font-family:sans-serif; padding:10px;'>
        <div style='font-size:18px; font-weight:bold; color:{color}; margin-bottom:8px;'>
            {label}
        </div>
        <div style='background:#eee; border-radius:8px; overflow:hidden; height:24px;'>
            <div style='width:{pct}%; background:{color}; height:100%;
                        display:flex; align-items:center; justify-content:center;
                        color:white; font-weight:bold; font-size:13px;'>
                {prob:.1%}
            </div>
        </div>
        <div style='margin-top:6px; font-size:13px; color:#555;'>
            Normal: {1-prob:.1%} &nbsp;|&nbsp; Pathological: {prob:.1%}
        </div>
    </div>
    """

Mounted at /content/drive
Device: cpu
Both models loaded


In [4]:
#General model inference ──────────────────────────────────────────
def predict_general(image):
    if image is None:
        return None, None, "<p>Please upload an image.</p>"

    img_pil = Image.fromarray(image).convert('RGB')
    prob_normal, prob_patho, img_np, tensor = run_inference(general_model, img_pil)

    label     = 'Pathological' if prob_patho >= 0.5 else 'Normal'
    conf_html = make_confidence_bar(prob_patho, label)
    _, cam_overlay = run_gradcampp(general_model, tensor, img_np)

    return img_np, cam_overlay, conf_html


# ─SPE model inference ───────────────────────────────────────────────
def predict_spe(image):
    if image is None:
        return None, None, "<p>Please upload an image.</p>"

    img_pil = Image.fromarray(image).convert('RGB')
    prob_normal, prob_patho, img_np, tensor = run_inference(spe_model, img_pil)

    label     = 'Peritoneal Endometriosis Detected' if prob_patho >= 0.5 else 'No Peritoneal Lesion'
    conf_html = make_confidence_bar(prob_patho, label if prob_patho >= 0.5 else 'Normal')
    _, cam_overlay = run_gradcampp(spe_model, tensor, img_np)

    return img_np, cam_overlay, conf_html

In [6]:
#Gradio UI ─────────────────────────────────────────────────────────
css = """
.gradio-container { font-family: 'Segoe UI', sans-serif; }
.tab-nav button { font-size: 15px; font-weight: bold; }
footer { display: none !important; }
"""

with gr.Blocks(css=css, title="GLENDA Endometriosis Detection") as demo:

    gr.Markdown("""
    #  GLENDA Endometriosis Detection
    Upload a laparoscopic image and get an instant prediction with visual explanation.
    """)

    with gr.Tabs():

        # ── Tab 1: General Model ──────────────────────────────────────────────
        with gr.Tab("General Endometriosis Model"):
            gr.Markdown("""
            ### General EdgeNeXt_Small
            Detects all four endometriosis classes:
            **Peritoneum · Ovary · TIE · Uterus**
            """)

            with gr.Row():
                with gr.Column(scale=1):
                    gen_input = gr.Image(
                        label="Upload Laparoscopic Frame",
                        type='numpy',
                        height=320
                    )
                    gen_btn = gr.Button(
                        "Analyse Image",
                        variant="primary",
                        size="lg"
                    )

                with gr.Column(scale=2):
                    gen_conf  = gr.HTML(label="Prediction & Confidence")
                    with gr.Row():
                        gen_orig = gr.Image(
                            label="Original (320×320)",
                            type='numpy',
                            height=300
                        )
                        gen_cam  = gr.Image(
                            label="GradCAM++ — Where the model is looking",
                            type='numpy',
                            height=300
                        )

            gen_btn.click(
                fn=predict_general,
                inputs=[gen_input],
                outputs=[gen_orig, gen_cam, gen_conf]
            )

            gr.Markdown("""
            >  **Heatmap guide:** Red = high model attention · Blue = low attention
            > The highlighted region shows which part of the image most influenced the prediction.
            """)

        # ── Tab 2: SPE Fine-tuned Model ───────────────────────────────────────
        with gr.Tab("SPE Peritoneal Fine-tuned Model"):
            gr.Markdown("""
            ### SPE Fine-tuned EdgeNeXt_Small
            Specialised for **peritoneal endometriosis** detection only.
            Fine-tuned from the general model using progressive layer unfreezing.
            """)

            with gr.Row():
                with gr.Column(scale=1):
                    spe_input = gr.Image(
                        label="Upload Laparoscopic Frame",
                        type='numpy',
                        height=320
                    )
                    spe_btn = gr.Button(
                        "Analyse Image",
                        variant="primary",
                        size="lg"
                    )

                with gr.Column(scale=2):
                    spe_conf  = gr.HTML(label="Prediction & Confidence")
                    with gr.Row():
                        spe_orig = gr.Image(
                            label="Original (320×320)",
                            type='numpy',
                            height=300
                        )
                        spe_cam  = gr.Image(
                            label="GradCAM++ — Where the model is looking",
                            type='numpy',
                            height=300
                        )

            spe_btn.click(
                fn=predict_spe,
                inputs=[spe_input],
                outputs=[spe_orig, spe_cam, spe_conf]
            )

            gr.Markdown("""
            > This model is specifically calibrated for peritoneal lesions.
            > Use the General Model tab for all other lesion types.
            """)

    gr.Markdown("""
    ---
    *EdgeNeXt_Small · GradCAM++ · GLENDA v1.5 · Patient-wise 5-fold CV*
    """)

demo.launch(share=True, debug=False)

/tmp/ipykernel_35702/77674858.py:8: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="GLENDA Endometriosis Detection") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://60dea763f279a1874b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
